
Project : Pentaho Log Intelligence

Layer   : Gold

Notebook: 05_Gold_logs_Table_analytics_ai

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Gold Delta table.

Author: Ernesto Felipe Garay Cervantes


#### Configuracion

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    current_timestamp
)

In [0]:
CATALOG = "pentaho_logs"
GOLD_TABLE_CARTE= "pentaho_logs.gold.gold_logs_carte"
GOLD_TABLE_CATALINA= "pentaho_logs.gold.gold_logs_catalina"
GOLD_TABLE_LOCALHOST= "pentaho_logs.gold.gold_logs_localhostaccess"
GOLD_TABLE_PENTAHO= "pentaho_logs.gold.gold_logs_pentaho"

#### Lectura  de tabla GOLD 

In [0]:
df_carte_gold = spark.table(GOLD_TABLE_CARTE)

#display(df_carte_gold.limit(20))

In [0]:
df_catalina_gold = spark.table(GOLD_TABLE_CATALINA)

#display(df_catalina_gold)

In [0]:
df_localhost_gold = spark.table(GOLD_TABLE_LOCALHOST)

#display(df_localhost_gold.limit(20))

In [0]:
df_pentaho_gold = spark.table(GOLD_TABLE_PENTAHO)

#display(df_pentaho_gold.limit(20))

- #### SELECCION DE CAMPOS 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

In [0]:
df_carte_gold_ai = df_carte_gold.select(
 col("event_id").alias("original_event_id"),
  col("file_name").alias("archivo_fuente"),
  col("application").alias("log_aplicacion"),
  col("server_port").alias("puerto"),
    F.expr("try_cast(NULLIF(TRIM(CAST(log_date AS STRING)), '') AS DATE)").alias("fecha_log"),
  col("tiempo_evento").alias("tiempo_evento"),
  col("Nivel_log").alias("nivel_log"),
  col("Descripcion").alias("Descripcion")
)

In [0]:
df_catalina_gold_ai = df_catalina_gold.select(
 col("event_id").alias("original_event_id"),
  col("file_name").alias("archivo_fuente"),
  col("application").alias("log_aplicacion"),
    F.expr("try_cast(NULLIF(TRIM(CAST(fecha AS STRING)), '') AS DATE)").alias("fecha_log"),
  col("hora").alias("tiempo_evento"),
  col("Nivel").alias("nivel_log"),
  col("mensaje").alias("Descripcion")
)


In [0]:
df_localhost_gold_ai = df_localhost_gold.select(
 col("event_id").alias("original_event_id"),
  col("file_name").alias("archivo_fuente"),
  col("aplicacion").alias("log_aplicacion"),
    F.expr("try_cast(NULLIF(TRIM(CAST(Fecha_evento AS STRING)), '') AS DATE)").alias("fecha_log"),
  col("hora").alias("tiempo_evento"),
  col("puerto").alias("puerto"),
  col("descripcion").alias("Descripcion")
  
  )

In [0]:
df_pentaho_gold_ai = df_pentaho_gold.select(
 col("event_id").alias("original_event_id"),
  col("file_name").alias("archivo_fuente"),
  col("application").alias("log_aplicacion"),
    F.expr("try_cast(NULLIF(TRIM(CAST(fecha AS STRING)), '') AS DATE)").alias("fecha_log"),
  col("hora").alias("tiempo_evento"),
  col("Nivel").alias("nivel_log"),
  col("descripcion").alias("Descripcion")
)


#### Union de datos en DATAFRAME

In [0]:
df_gold_analytics_ai = (
df_carte_gold_ai
.unionByName(df_catalina_gold_ai,allowMissingColumns=True)
.unionByName(df_localhost_gold_ai,allowMissingColumns=True)
.unionByName(df_pentaho_gold_ai,allowMissingColumns=True)
)


#display(df_gold_analytics_ai)


#### validación DATAFRAME

In [0]:
print("CARTE:", df_carte_gold.count())
print("CATALINA:", df_catalina_gold.count())
print("LOCALHOST:", df_localhost_gold.count())
print("PENTAHO:", df_pentaho_gold.count())

#### Creación Tabla GOLD Pentaho_Log

In [0]:
from pyspark.sql.functions import sha2, concat_ws, col

df_gold_analytics_ai = df_gold_analytics_ai.withColumn(
    "event_id_analytics",
    sha2(
        concat_ws(
            "||",
            col("archivo_fuente"),
            col("original_event_id")
        ),
        256
    )
)

In [0]:
df_gold_analytics_ai = df_gold_analytics_ai.select(
"event_id_analytics",
"archivo_fuente",
"log_aplicacion",
"puerto",
"fecha_log",
"tiempo_evento",
"nivel_log",
"Descripcion"
)

#### Validación datos nuevos

In [0]:
GOLD_TABLE_IA_ANALYTICS = "pentaho_logs.gold.gold_logs_analyticsia"
#falta validar

#validacion
if spark.catalog.tableExists(GOLD_TABLE_IA_ANALYTICS):
  # Registros que ya existen en Analytics
    df_analytics_existente = (
        spark.table(GOLD_TABLE_IA_ANALYTICS)
        .select("event_id_analytics")
        .distinct()
    )

    # Nos quedamos solamente con registros nuevos
    df_gold_analytics_nuevos = (
        df_gold_analytics_ai.join(
            df_analytics_existente,
            on="event_id_analytics",
            how="left_anti"
        )
    )


else:
    # Si la tabla aún no existe, todos los registros son nuevos
    df_gold_analytics_nuevos = df_gold_analytics_ai


#### CREACIÓN TABLA GOLD ANALYTICS


In [0]:
GOLD_TABLE_IA_ANALYTICS = "pentaho_logs.gold.gold_logs_analyticsIA"
(
    df_gold_analytics_ai.write
        .format("delta")
        .mode("append") #cambia overwrite por append
        .saveAsTable(GOLD_TABLE_IA_ANALYTICS)
)

In [0]:
#display(spark.table(GOLD_TABLE_IA_ANALYTICS).limit(20))